# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [3]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_DEEPSEEK = 'deepseek-chat'

In [4]:
# set up environment
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('DEEPSEEK_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'deepseek-chat'
openai = OpenAI(base_url="https://api.deepseek.com/v1", api_key=api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [14]:
# set up environment
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('DEEPSEEK_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'deepseek-chat'
llama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [5]:
system_prompt ="""You are an expert coding assistant and LLM engineer. 
Whatever I ask, explain clearly and concisely. 
Use simple examples or analogies to make your answers easy to understand."""

print(system_prompt)

You are an expert coding assistant and LLM engineer. 
Whatever I ask, explain clearly and concisely. 
Use simple examples or analogies to make your answers easy to understand.


In [6]:
# here is the question; type over this to ask something new

user_prompt = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [17]:
# Get gpt-4o-mini to answer, with streaming
def explain(system_prompt, user_prompt):
    stream = openai.chat.completions.create(
        model=MODEL_DEEPSEEK,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [18]:
# Get gpt-4o-mini to answer, with streaming
def explain_llama(system_prompt, user_prompt):
    stream = llama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
#DEEPSEEK
explain(system_prompt, user_prompt)

This code does two things:

**What it does:**
It yields each unique author name from a list of book dictionaries, but only if the book has an "author" field.

**Breaking it down:**

1. **The comprehension part** `{book.get("author") for book in books if book.get("author")}`
   - Creates a **set** (notice the curly braces `{}`)
   - Iterates over `books` collection
   - For each book, gets the author using `.get("author")` (returns `None` instead of error if missing)
   - The `if` condition filters out books without an author
   - Being a set, it automatically removes duplicate authors

2. **The `yield from`** 
   - Makes this function a generator
   - Delegates to the set's iterator
   - Yields each author one at a time, on demand

**Why use this pattern?**

```python
# Instead of:
def get_authors(books):
    authors = set()
    for book in books:
        if book.get("author"):
            authors.add(book.get("author"))
    return authors  # Returns all at once, wasting memory

# Use this:
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}
    # Lazily yields authors, less memory

# Usage:
for author in get_authors(books):
    print(author)
```

**Key benefits:**
- **Memory efficient**: Doesn't need to store all books before processing
- **Duplicate free**: Set automatically removes duplicates
- **Safe**: `.get()` prevents KeyError if "author" is missing
- **Concise**: Single line does iteration, filtering, deduplication, and yielding

The main "why" is **lazy evaluation** - you process and yield results only as needed, which is especially useful with large datasets.

In [19]:
#LLAMA
explain_llama(system_prompt, user_prompt)

Let's break down the code step by step:

**What is it doing?**

This code is using a concept called "generator yields" to extract data from an iterable.

Here's what happens:

1. `yield from`: This keyword is used to delegate the responsibility of yielding values to another iterator (or generator) inside this function.
2. `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension that iterates over an iterable (`books`), filters it based on a condition, and extracts values.

**How does it work?**

Imagine you have a list of bookstore objects (`books`) with each object having an `author` attribute. You want to extract the authors from all the books in this list.

The dictionary comprehension works as follows:

1. Iterate over each book object in `books`.
2. For each book, check if it has an `author` attribute using `book.get("author")`. This is a safety check to ensure that only valid objects with the `author` attribute are processed.
3. If the book has an author, extract the author's name from the book object (e.g., `"John Doe"`).

The generator uses `yield from` to delegate this processing to another iterator (which would be created by the dictionary comprehension itself). This allows us to avoid storing all the data in memory at once.

**Outcome**

The `yield from ...` expression will yield each author's name, one by one. The receiver of this code (typically a coroutine or an asynchronous function) can then choose whether to iterate over and use these values, without having to store them all initially.

Here's a simple example to illustrate the difference between storing data in memory vs. yielding values:

**Before `yield from`:**
```python
books = [Book1("John Doe"), Book2("Jane Smith")]
authors = []
for book in books:
    authors.extend(book.author)
```
Stores all authors in memory (`authors` list).

**After `yield from`:**
```python
def get_authors():
    for book in books if book.get("author"):
        yield book.get("author")

authors = []
for author in get_authors():
    authors.append(author)
```
Yields each author's name one by one, without storing them all initially.

I hope this explanation helps!